# cMSCI GPU Training Notebook

**Self-contained notebook for training Variant E (CrossSpaceBridge) and Variant F (ProbVLM adapters)**

Upload this notebook + `combined_training.npz` (7.5 MB) to your JupyterHub.

Then: **Runtime → Run All** or run cell-by-cell.

**Expected time: ~5-15 minutes on A6000**

## Cell 1: Verify GPU

In [ ]:
import torch
import numpy as np
import json
import time
from pathlib import Path

print("="*60)
print("GPU CHECK")
print("="*60)

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {gpu_name}")
    print(f"  VRAM: {gpu_mem:.1f} GB")
    print(f"  CUDA version: {torch.version.cuda}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"
    print("  Using Apple MPS")
else:
    device = "cpu"
    print("  WARNING: No GPU found! Training will be slow.")

print(f"  PyTorch: {torch.__version__}")
print(f"  Device: {device}")
print("="*60)

## Cell 2: Load Training Data

Make sure `combined_training.npz` is in the **same folder** as this notebook.

If you put it somewhere else, change the path below.

In [ ]:
# === CHANGE THIS PATH IF NEEDED ===
DATA_PATH = "combined_training.npz"
# ==================================

if not Path(DATA_PATH).exists():
    # Try common alternative locations
    for alt in ["data/bridge_training/combined_training.npz", 
                "../data/bridge_training/combined_training.npz",
                "~/combined_training.npz"]:
        if Path(alt).expanduser().exists():
            DATA_PATH = str(Path(alt).expanduser())
            break

data = np.load(DATA_PATH)
image_embs = data["image_embeddings"]  # CLIP image embeddings (N, 512)
audio_embs = data["audio_embeddings"]  # CLAP audio embeddings (N, 512)

print(f"Loaded training data from: {DATA_PATH}")
print(f"  Image embeddings: {image_embs.shape} {image_embs.dtype}")
print(f"  Audio embeddings: {audio_embs.shape} {audio_embs.dtype}")
print(f"  Total pairs: {len(image_embs)}")

# Quick sanity check
assert image_embs.shape[1] == 512, f"Expected 512-d CLIP, got {image_embs.shape[1]}"
assert audio_embs.shape[1] == 512, f"Expected 512-d CLAP, got {audio_embs.shape[1]}"
print("  Sanity check passed!")

## Cell 3: Define Models

All model code is included here — no external dependencies needed beyond PyTorch.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

# ============================================================
# CrossSpaceBridge — Maps CLIP images + CLAP audio → shared space
# ============================================================

class BridgeProjectionHead(nn.Module):
    """Single projection: Linear→GELU→Dropout→Linear→L2norm"""
    def __init__(self, input_dim=512, hidden_dim=384, output_dim=256, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, bias=True),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim, bias=False),
        )
    def forward(self, x):
        return F.normalize(self.net(x), p=2, dim=-1)


class CrossSpaceBridge(nn.Module):
    """Learned bridge: CLIP image space ↔ CLAP audio space via shared 256-d space."""
    def __init__(self, clip_dim=512, clap_dim=512, bridge_dim=256, hidden_dim=384, dropout=0.1):
        super().__init__()
        self.image_proj = BridgeProjectionHead(clip_dim, hidden_dim, bridge_dim, dropout)
        self.audio_proj = BridgeProjectionHead(clap_dim, hidden_dim, bridge_dim, dropout)
        self.config = dict(clip_image_dim=clip_dim, clap_audio_dim=clap_dim,
                           bridge_dim=bridge_dim, hidden_dim=hidden_dim, dropout=dropout)

    def forward(self, image_emb=None, audio_emb=None):
        result = {}
        if image_emb is not None: result["image"] = self.image_proj(image_emb)
        if audio_emb is not None: result["audio"] = self.audio_proj(audio_emb)
        return result

    def compute_similarity(self, image_emb_np, audio_emb_np):
        """Numpy interface: compute image-audio similarity through bridge."""
        self.eval()
        with torch.no_grad():
            img = torch.tensor(image_emb_np, dtype=torch.float32).unsqueeze(0)
            aud = torch.tensor(audio_emb_np, dtype=torch.float32).unsqueeze(0)
            proj = self.forward(image_emb=img, audio_emb=aud)
            return float(F.cosine_similarity(proj["image"], proj["audio"]).item())

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), path)
        with path.with_suffix('.json').open('w') as f:
            json.dump(self.config, f, indent=2)

    @classmethod
    def load(cls, path):
        path = Path(path)
        with path.with_suffix('.json').open('r') as f:
            config = json.load(f)
        model = cls(**config)
        model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
        return model.eval()


# ============================================================
# ProbabilisticAdapter — BayesCap/ProbVLM uncertainty estimation
# ============================================================

class ProbabilisticAdapter(nn.Module):
    """Predicts Generalized Gaussian parameters (mu, alpha, beta) from an embedding."""
    def __init__(self, input_dim=512, hidden_dim=256, num_layers=3, dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        layers = []
        in_d = input_dim
        for _ in range(num_layers - 1):
            layers.extend([nn.Linear(in_d, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
            in_d = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.mu_head = nn.Linear(hidden_dim, input_dim)
        self.alpha_head = nn.Linear(hidden_dim, input_dim)
        self.beta_head = nn.Linear(hidden_dim, input_dim)
        self.config = dict(input_dim=input_dim, hidden_dim=hidden_dim,
                           num_layers=num_layers, dropout=dropout)

    def forward(self, embedding):
        h = self.backbone(embedding)
        mu = embedding + self.mu_head(h)  # Residual connection
        alpha = F.softplus(self.alpha_head(h)) + 1e-6
        beta = F.softplus(self.beta_head(h)) + 1e-6
        return mu, alpha, beta

    def uncertainty(self, embedding_np):
        """Scalar uncertainty = mean predicted alpha."""
        self.eval()
        emb = embedding_np.squeeze()
        if emb.ndim == 1: emb = emb[np.newaxis, :]
        with torch.no_grad():
            _, alpha, _ = self.forward(torch.tensor(emb, dtype=torch.float32))
            return float(alpha.mean().item())

    def save(self, path):
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), p)
        with p.with_suffix('.json').open('w') as f: json.dump(self.config, f, indent=2)

    @classmethod
    def load(cls, path):
        p = Path(path)
        with p.with_suffix('.json').open('r') as f: config = json.load(f)
        model = cls(**config)
        model.load_state_dict(torch.load(p, map_location='cpu', weights_only=True))
        return model.eval()


print("Models defined successfully!")
print(f"  CrossSpaceBridge: 512→256 bridge space")
print(f"  ProbabilisticAdapter: 512→(mu, alpha, beta)")

## Cell 4: Define Training Components

In [ ]:
# ============================================================
# Datasets
# ============================================================

class ImageAudioPairDataset(Dataset):
    """Paired (CLIP image, CLAP audio) embeddings for bridge training."""
    def __init__(self, image_embeddings, audio_embeddings):
        assert len(image_embeddings) == len(audio_embeddings)
        self.images = torch.tensor(image_embeddings, dtype=torch.float32)
        self.audio = torch.tensor(audio_embeddings, dtype=torch.float32)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx): return {"image": self.images[idx], "audio": self.audio[idx]}


class EmbeddingPairDataset(Dataset):
    """(input, target) embedding pairs for prob adapter training."""
    def __init__(self, inputs, targets):
        assert len(inputs) == len(targets)
        self.inputs = torch.tensor(inputs, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx): return self.inputs[idx], self.targets[idx]


# ============================================================
# Losses
# ============================================================

class BridgeInfoNCELoss(nn.Module):
    """Symmetric InfoNCE (same loss structure as CLIP itself)."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.tensor(np.log(1.0 / temperature)))

    @property
    def temperature(self):
        return torch.exp(-self.log_temperature)

    def forward(self, image_emb, audio_emb):
        batch_size = image_emb.size(0)
        logits = torch.mm(image_emb, audio_emb.t()) / self.temperature
        labels = torch.arange(batch_size, device=logits.device)
        loss_i2a = F.cross_entropy(logits, labels)
        loss_a2i = F.cross_entropy(logits.t(), labels)
        loss = 0.5 * (loss_i2a + loss_a2i)
        with torch.no_grad():
            acc_i2a = (logits.argmax(dim=1) == labels).float().mean()
            acc_a2i = (logits.t().argmax(dim=1) == labels).float().mean()
        return loss, {"loss": loss.item(), "acc_i2a": acc_i2a.item(),
                      "acc_a2i": acc_a2i.item(), "temp": self.temperature.item()}


class GenGaussNLL(nn.Module):
    """Negative log-likelihood for Generalized Gaussian distribution."""
    def forward(self, mu, alpha, beta, target):
        residual = torch.abs(target - mu)
        alpha_c = torch.clamp(alpha, min=1e-6)
        nll = torch.log(alpha_c) + (residual / alpha_c).pow(beta)
        return nll.mean()


print("Training components defined!")

## Cell 5: Train CrossSpaceBridge (Variant E)

This trains the bridge that maps CLIP image embeddings and CLAP audio embeddings into a shared 256-d space.

**Expected: ~2-5 minutes on GPU**

In [ ]:
print("="*60)
print("TRAINING: CrossSpaceBridge")
print("="*60)

# Hyperparameters
BRIDGE_EPOCHS = 50
BRIDGE_BATCH_SIZE = 64
BRIDGE_LR = 3e-4
BRIDGE_PATIENCE = 10
VAL_SPLIT = 0.15

# Output directory
BRIDGE_DIR = Path("trained_models/bridge")
BRIDGE_DIR.mkdir(parents=True, exist_ok=True)

# Build dataset
full_dataset = ImageAudioPairDataset(image_embs, audio_embs)
n_val = max(1, int(len(full_dataset) * VAL_SPLIT))
n_train = len(full_dataset) - n_val
train_data, val_data = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_data, batch_size=BRIDGE_BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_data, batch_size=BRIDGE_BATCH_SIZE, shuffle=False)

# Build model
bridge = CrossSpaceBridge().to(device)
loss_fn = BridgeInfoNCELoss().to(device)
optimizer = torch.optim.AdamW(
    list(bridge.parameters()) + list(loss_fn.parameters()),
    lr=BRIDGE_LR, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=BRIDGE_EPOCHS)

n_params = sum(p.numel() for p in bridge.parameters())
print(f"  Train: {n_train}, Val: {n_val}")
print(f"  Parameters: {n_params:,}")
print(f"  Device: {device}")
print()

# Training loop
best_val_loss = float("inf")
patience_counter = 0
history = []
t_start = time.time()

for epoch in range(BRIDGE_EPOCHS):
    # Train
    bridge.train(); loss_fn.train()
    epoch_metrics = []
    for batch in train_loader:
        img = batch["image"].to(device)
        aud = batch["audio"].to(device)
        optimizer.zero_grad()
        proj = bridge(image_emb=img, audio_emb=aud)
        loss, metrics = loss_fn(proj["image"], proj["audio"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bridge.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_metrics.append(metrics)
    scheduler.step()

    avg = {k: np.mean([m[k] for m in epoch_metrics]) for k in epoch_metrics[0]}

    # Validate
    bridge.eval(); loss_fn.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            img = batch["image"].to(device)
            aud = batch["audio"].to(device)
            proj = bridge(image_emb=img, audio_emb=aud)
            vloss, _ = loss_fn(proj["image"], proj["audio"])
            val_losses.append(vloss.item())
    val_loss = np.mean(val_losses)

    # Log
    avg["val_loss"] = val_loss
    history.append(avg)
    print(f"  Epoch {epoch+1:3d}/{BRIDGE_EPOCHS}: loss={avg['loss']:.4f}  "
          f"acc_i2a={avg['acc_i2a']:.3f}  acc_a2i={avg['acc_a2i']:.3f}  "
          f"val_loss={val_loss:.4f}  temp={avg['temp']:.3f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        bridge.save(BRIDGE_DIR / "bridge_best.pt")
    else:
        patience_counter += 1
        if patience_counter >= BRIDGE_PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

# Save final
bridge.save(BRIDGE_DIR / "bridge_final.pt")
with (BRIDGE_DIR / "bridge_training_history.json").open("w") as f:
    json.dump(history, f, indent=2)

elapsed = time.time() - t_start
print(f"\n  Bridge training complete in {elapsed:.1f}s")
print(f"  Best val_loss: {best_val_loss:.4f}")
print(f"  Saved: {BRIDGE_DIR / 'bridge_best.pt'}")

## Cell 6: Validate Bridge

Check that the bridge produces meaningful image-audio similarities: matched pairs should score higher than mismatched.

In [ ]:
print("="*60)
print("VALIDATION: CrossSpaceBridge")
print("="*60)

# Load best model
bridge_best = CrossSpaceBridge.load(BRIDGE_DIR / "bridge_best.pt")

# Compute matched vs mismatched similarities
n_val = min(200, len(image_embs))
rng = np.random.default_rng(42)
idx = rng.choice(len(image_embs), n_val, replace=False)

matched_sims = []
mismatched_sims = []

for i in range(n_val):
    # Matched: image[i] with its paired audio[i]
    sim_match = bridge_best.compute_similarity(image_embs[idx[i]], audio_embs[idx[i]])
    matched_sims.append(sim_match)
    
    # Mismatched: image[i] with random audio[j]
    j = rng.choice(len(audio_embs))
    while j == idx[i]: j = rng.choice(len(audio_embs))  # Ensure different
    sim_mismatch = bridge_best.compute_similarity(image_embs[idx[i]], audio_embs[j])
    mismatched_sims.append(sim_mismatch)

matched_arr = np.array(matched_sims)
mismatched_arr = np.array(mismatched_sims)
separation = np.mean(matched_arr) - np.mean(mismatched_arr)

print(f"  Matched pairs:    mean={np.mean(matched_arr):.4f}  std={np.std(matched_arr):.4f}")
print(f"  Mismatched pairs: mean={np.mean(mismatched_arr):.4f}  std={np.std(mismatched_arr):.4f}")
print(f"  Separation:       {separation:.4f}")
print()

if separation > 0.05:
    print("  PASS: Bridge learned meaningful image-audio alignment!")
elif separation > 0:
    print("  MARGINAL: Some separation, but could be better.")
else:
    print("  FAIL: No separation — bridge may need more data or tuning.")

## Cell 7: Train Probabilistic Adapters (Variant F)

Trains two ProbVLM-style adapters that estimate uncertainty:
- **CLIP adapter**: Trained on image embeddings
- **CLAP adapter**: Trained on audio embeddings

Each adapter learns to predict distribution parameters (mu, alpha, beta) for a Generalized Gaussian.

**Expected: ~3-8 minutes on GPU**

In [ ]:
ADAPTER_DIR = Path("trained_models/prob_adapters")
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

ADAPTER_EPOCHS = 100
ADAPTER_BATCH_SIZE = 32
ADAPTER_LR = 1e-4
ADAPTER_PATIENCE = 15


def train_probabilistic_adapter(embeddings, name, output_path):
    """Train a ProbVLM adapter on embeddings (self-supervised)."""
    print(f"\n{'='*60}")
    print(f"TRAINING: {name} Probabilistic Adapter")
    print(f"{'='*60}")
    
    # Self-supervised: input = target (adapter learns to model the embedding distribution)
    # Add small noise to targets for regularization
    rng = np.random.default_rng(42)
    noise = rng.normal(0, 0.01, size=embeddings.shape).astype(np.float32)
    targets = embeddings + noise
    
    dataset = EmbeddingPairDataset(embeddings, targets)
    n_val = max(1, int(len(dataset) * 0.15))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )
    
    train_loader = DataLoader(train_ds, batch_size=ADAPTER_BATCH_SIZE, shuffle=True,
                              drop_last=len(train_ds) > ADAPTER_BATCH_SIZE)
    val_loader = DataLoader(val_ds, batch_size=ADAPTER_BATCH_SIZE, shuffle=False)
    
    adapter = ProbabilisticAdapter(input_dim=512).to(device)
    optimizer = torch.optim.AdamW(adapter.parameters(), lr=ADAPTER_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ADAPTER_EPOCHS)
    
    l1_loss = nn.L1Loss()
    gg_loss = GenGaussNLL()
    
    n_params = sum(p.numel() for p in adapter.parameters())
    print(f"  Train: {n_train}, Val: {n_val}")
    print(f"  Parameters: {n_params:,}")
    
    best_val_loss = float("inf")
    patience_counter = 0
    t_start = time.time()
    
    for epoch in range(ADAPTER_EPOCHS):
        adapter.train()
        train_losses = []
        for inp, tgt in train_loader:
            inp, tgt = inp.to(device), tgt.to(device)
            optimizer.zero_grad()
            mu, alpha, beta = adapter(inp)
            loss = l1_loss(mu, tgt) + gg_loss(mu, alpha, beta, tgt)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()
        
        # Validate
        adapter.eval()
        val_losses = []
        with torch.no_grad():
            for inp, tgt in val_loader:
                inp, tgt = inp.to(device), tgt.to(device)
                mu, alpha, beta = adapter(inp)
                loss = l1_loss(mu, tgt) + gg_loss(mu, alpha, beta, tgt)
                val_losses.append(loss.item())
        
        avg_train = np.mean(train_losses)
        avg_val = np.mean(val_losses)
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{ADAPTER_EPOCHS}: train={avg_train:.4f}  val={avg_val:.4f}")
        
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            patience_counter = 0
            adapter.save(output_path)
        else:
            patience_counter += 1
            if patience_counter >= ADAPTER_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break
    
    elapsed = time.time() - t_start
    print(f"  {name} training complete in {elapsed:.1f}s (best val={best_val_loss:.4f})")
    
    # Load best and return
    adapter = ProbabilisticAdapter.load(output_path).to(device)
    return adapter


# Train CLIP adapter (on image embeddings)
clip_adapter = train_probabilistic_adapter(
    image_embs, "CLIP (Image)", str(ADAPTER_DIR / "clip_adapter.pt")
)

# Train CLAP adapter (on audio embeddings)
clap_adapter = train_probabilistic_adapter(
    audio_embs, "CLAP (Audio)", str(ADAPTER_DIR / "clap_adapter.pt")
)

## Cell 8: Validate Probabilistic Adapters

In [ ]:
print("="*60)
print("VALIDATION: Probabilistic Adapters")
print("="*60)

# Check uncertainty distributions
n_check = min(100, len(image_embs))

clip_uncertainties = [clip_adapter.uncertainty(image_embs[i]) for i in range(n_check)]
clap_uncertainties = [clap_adapter.uncertainty(audio_embs[i]) for i in range(n_check)]

print(f"  CLIP adapter uncertainty:  mean={np.mean(clip_uncertainties):.6f}  "
      f"std={np.std(clip_uncertainties):.6f}  "
      f"range=[{np.min(clip_uncertainties):.6f}, {np.max(clip_uncertainties):.6f}]")
print(f"  CLAP adapter uncertainty:  mean={np.mean(clap_uncertainties):.6f}  "
      f"std={np.std(clap_uncertainties):.6f}  "
      f"range=[{np.min(clap_uncertainties):.6f}, {np.max(clap_uncertainties):.6f}]")

# Check that uncertainty varies (not constant)
clip_cv = np.std(clip_uncertainties) / (np.mean(clip_uncertainties) + 1e-10)
clap_cv = np.std(clap_uncertainties) / (np.mean(clap_uncertainties) + 1e-10)

print(f"\n  CLIP coefficient of variation: {clip_cv:.4f}")
print(f"  CLAP coefficient of variation: {clap_cv:.4f}")

if clip_cv > 0.01 and clap_cv > 0.01:
    print("\n  PASS: Both adapters produce varying uncertainty estimates!")
else:
    print("\n  WARNING: Low variation in uncertainty — adapters may need more diverse training data.")

## Cell 9: Summary & Download Instructions

In [ ]:
print("="*60)
print("ALL TRAINING COMPLETE!")
print("="*60)
print()
print("Trained models saved to:")
print(f"  {BRIDGE_DIR / 'bridge_best.pt'}")
print(f"  {BRIDGE_DIR / 'bridge_best.json'}")
print(f"  {BRIDGE_DIR / 'bridge_final.pt'}")
print(f"  {BRIDGE_DIR / 'bridge_final.json'}")
print(f"  {BRIDGE_DIR / 'bridge_training_history.json'}")
print(f"  {ADAPTER_DIR / 'clip_adapter.pt'}")
print(f"  {ADAPTER_DIR / 'clip_adapter.json'}")
print(f"  {ADAPTER_DIR / 'clap_adapter.pt'}")
print(f"  {ADAPTER_DIR / 'clap_adapter.json'}")
print()
print("File sizes:")
for p in sorted(Path("trained_models").rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p}: {size_kb:.1f} KB")
print()
print("="*60)
print("NEXT STEPS")
print("="*60)
print()
print("1. Download the 'trained_models/' folder to your laptop")
print("2. Copy files into your project:")
print("     trained_models/bridge/bridge_best.pt   → models/bridge/bridge_best.pt")
print("     trained_models/bridge/bridge_best.json  → models/bridge/bridge_best.json")
print("     trained_models/prob_adapters/clip_adapter.pt  → models/prob_adapters/clip_adapter.pt")
print("     trained_models/prob_adapters/clip_adapter.json → models/prob_adapters/clip_adapter.json")
print("     trained_models/prob_adapters/clap_adapter.pt  → models/prob_adapters/clap_adapter.pt")
print("     trained_models/prob_adapters/clap_adapter.json → models/prob_adapters/clap_adapter.json")
print()
print("3. On your laptop, run:")
print("     python scripts/run_cmsci_comparison.py --all")
print("     python scripts/run_cmsci_ablation.py")
print("     python scripts/generate_paper_figures.py")

## Cell 10 (Optional): Create ZIP for Easy Download

In [ ]:
import shutil

zip_path = shutil.make_archive("trained_models", "zip", ".", "trained_models")
zip_size = Path(zip_path).stat().st_size / (1024 * 1024)
print(f"Created: {zip_path} ({zip_size:.1f} MB)")
print("\nDownload this single ZIP file, then unzip on your laptop.")